In [5]:
import torch
import numpy as np

# Creating a Tensor

In [10]:
data = [[1, 2],[3, 4]]

# Creating directly from data
x_data = torch.tensor(data)


# Creating from numpy array
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

# Creating from another tensor
x_ones = torch.ones_like(x_data)  # retains the properties of x_data

x_rand = torch.rand_like(x_data, dtype=torch.float)  # overrides the dtype

In [9]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.2547, 0.7320, 0.2061],
        [0.5142, 0.5133, 0.8522]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


# Tensor Attributes

In [11]:
tensor = torch.rand(3, 4)

# shape
print(f"Shape of tensor: {tensor.shape}")
# dtype
print(f"Data type of tensor: {tensor.dtype}")
# device
print(f"Device of tensor: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Data type of tensor: torch.float32
Device of tensor: cpu


In [13]:
torch.accelerator.is_available() # Check if a GPU is available

False

# Tensor Operations

In [18]:
tensor

tensor([[0.2394, 0.8806, 0.9899, 0.5590],
        [0.4071, 0.6290, 0.5154, 0.5826],
        [0.6229, 0.4619, 0.9644, 0.5468]])

In [23]:
tensor[0] # Row 1

tensor([0.2394, 0.8806, 0.9899, 0.5590])

In [22]:
tensor[..., -1] # In Python the '...' means "all previous dimensions"

tensor([0.5590, 0.5826, 0.5468])

In [27]:
cat = torch.cat([tensor, tensor], dim=1)
cat

tensor([[0.2394, 0.8806, 0.9899, 0.5590, 0.2394, 0.8806, 0.9899, 0.5590],
        [0.4071, 0.6290, 0.5154, 0.5826, 0.4071, 0.6290, 0.5154, 0.5826],
        [0.6229, 0.4619, 0.9644, 0.5468, 0.6229, 0.4619, 0.9644, 0.5468]])

In [29]:
# if you have a single element tensor (e.g. after an aggregation) you can convert it
# to a Python value using item()
print(f"Aggregation: {tensor.sum()}")
print(f"Item: {tensor.sum().item()}")

Aggregation: 7.398948669433594
Item: 7.398948669433594


# Datasets & DataLoaders

`Dataset` stores the samples and their corresponding labels, and `DataLoader` wraps an iterable around the `Dataset` to enable easy access to the samples.

In [31]:
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

In [32]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100.0%
100.0%
100.0%
100.0%


In [ ]:
labels_map = {
    0: "T-Shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle Boot",
}
figure = plt.figure(figsize=(8, 8))
cols, rows = 3, 3
for i in range(1, cols * rows + 1):
    sample_idx = torch.randint(len(training_data), size=(1,)).item()
    img, label = training_data[sample_idx]
    figure.add_subplot(rows, cols, i)
    plt.title(labels_map[label])
    plt.axis("off")
    plt.imshow(img.squeeze(), cmap="gray")

## Custom Datasets

A custom Dataset class must implement three functions:
- `__init__`
- `__len__`
- `__getitem__`

In [46]:
from typing import List, Tuple, Any

class CustomDataset(Dataset):
    def __init__(self, features: List[Any], labels: List[Any]):
        self.features = features
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)
    
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        data = torch.tensor(self.features[index])
        label = torch.tensor(self.labels[index])
        return data, label

# Example usage
features = [1, 2, 3, 4, 5]
labels = [1, 2, 3, 4, 5]
dataset = CustomDataset(features, labels)
print(dataset[0])  # Output: (tensor(1), tensor(1))
print(len(dataset))  # Output: 5

(tensor(1), tensor(1))
5


The `Dataset` retrieves samples one at a time. `DataLoader` is an iterable that allows typical batching done with deep learning

In [48]:
from torch.utils.data import DataLoader

# Continuing using the FashionMNIST data
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)

In [49]:
train_features, train_labels = next(iter(train_dataloader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")

Feature batch shape: torch.Size([64, 1, 28, 28])
Labels batch shape: torch.Size([64])


In [ ]:
img = train_features[0].squeeze()
label = train_labels[0]
plt.imshow(img, cmap="gray")
plt.show()
print(f"Label: {labels_map[label.item()]}")